In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!pip install tqdm

In [ ]:
# @title Manga Upscaler + Downscaler (Per-Image Timer) { display-mode: "form" }

import os
from google.colab import drive
import subprocess
from pathlib import Path
import torch
from PIL import Image
import shutil
import time

# ==== ✅ CUSTOM FOLDERS ====
BW_INPUT_FOLDER = "/content/gdrive/MyDrive/ESRGAN (1)/bw/Vol.01 Ch.0002 - Rope Partner (en) [Bakana Haven]" # @param {type:"string"}
COLOR_INPUT_FOLDER = "/content/gdrive/MyDrive/ESRGAN/color" # @param {type:"string"}
FINAL_OUTPUT_FOLDER = "/content/gdr" # @param {type:"string"}
# ==== CORE SETUP ====
def check_connect_gdrive():
    if not os.path.exists("/content/gdrive/MyDrive"):
        print("🔌 Mounting Google Drive...")
        drive.mount("/content/gdrive")

def check_clone_esrgan():
    if not os.path.exists("ESRGAN"):
        print("⬇️ Cloning ESRGAN...")
        subprocess.run(["git", "clone", "https://github.com/Spladenly/ESRGAN"])

def init_dirs():
    Path(FINAL_OUTPUT_FOLDER).mkdir(parents=True, exist_ok=True)
    Path("/content/temp_input").mkdir(parents=True, exist_ok=True)
    Path("/content/temp_output").mkdir(parents=True, exist_ok=True)

def dir_contains_files(path):
    return os.path.exists(path) and any(Path(path).glob("*"))

def upscale_and_downscale_image(image_path, model_path):
    temp_input = "/content/temp_input"
    temp_output = "/content/temp_output"

    # Clear temp folders
    for folder in [temp_input, temp_output]:
        for file in Path(folder).glob("*"):
            file.unlink()

    # Copy image to temp input
    shutil.copy(image_path, temp_input)

    # Run ESRGAN on the image
    subprocess.run([
        "python", "ESRGAN/upscale.py", "-se",
        "-i", temp_input,
        "-o", temp_output,
        model_path
    ], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

    # Resize immediately
    filename = os.path.basename(image_path)
    upscaled_path = os.path.join(temp_output, filename)
    final_output_path = os.path.join(FINAL_OUTPUT_FOLDER, filename)

    if os.path.exists(upscaled_path):
        with Image.open(upscaled_path) as img:
            new_size = (img.width // 2, img.height // 2)
            resized = img.resize(new_size, Image.LANCZOS)
            resized.save(final_output_path)
            return True
    return False

def process_folder(folder_path, model_path):
    print(f"\n📁 Processing: {folder_path}")
    exts = [".jpg", ".jpeg", ".png", ".bmp", ".webp"]
    images = [f for f in os.listdir(folder_path) if Path(f).suffix.lower() in exts]

    total = len(images)
    for idx, file in enumerate(images, 1):
        image_path = os.path.join(folder_path, file)
        start = time.time()

        success = upscale_and_downscale_image(image_path, model_path)

        end = time.time()
        duration = end - start
        if success:
            print(f"✅ {file} ({idx}/{total}) | ⏱ {duration:.2f} sec")
        else:
            print(f"❌ Failed: {file} ({idx}/{total})")

def main():
    print("📈 Manga Upscaler + 2x Downscaler with Per-Image Timing")

    if not torch.cuda.is_available():
        print("❌ GPU not enabled. Set 'Runtime' → 'Change runtime type' → GPU.")
        return

    check_connect_gdrive()
    check_clone_esrgan()
    init_dirs()

    if dir_contains_files(BW_INPUT_FOLDER):
        print("🎨 Upscaling B&W images...")
        process_folder(BW_INPUT_FOLDER, "ESRGAN/models/4x_eula_digimanga_bw_v2_nc1_307k.pth")

    if dir_contains_files(COLOR_INPUT_FOLDER):
        print("🎨 Upscaling Color images...")
        process_folder(COLOR_INPUT_FOLDER, "ESRGAN/models/4x-AnimeSharp.pth")

    print(f"\n📁 Done. Final output saved to:\n{FINAL_OUTPUT_FOLDER}")

if __name__ == "__main__":
    main()


In [ ]:
# @title Manga Upscaler (2x-MangaScaleV3 Batch) 🚀 { display-mode: "form" }

import os, shutil, subprocess, time
from pathlib import Path
from google.colab import drive
import torch

# ==== FOLDERS ====
INPUT_FOLDER = "/content/gdrive/MyDrive/ESRGAN/bw" # @param {type:"string"}
OUTPUT_FOLDER = "/content/gd" # @param {type:"string"}

# ==== SETUP ====
def mount_drive():
    if not os.path.exists("/content/gdrive/MyDrive"):
        print("🔌 Mounting Google Drive...")
        drive.mount("/content/gdrive")

def clone_esrgan():
    if not os.path.exists("ESRGAN"):
        print("⬇️ Cloning ESRGAN repo...")
        subprocess.run(["git", "clone", "https://github.com/Spladenly/ESRGAN"], check=True)

def download_model():
    model_url = "https://huggingface.co/NoCrypt/mangascale/resolve/main/2x_MangaScaleV3.pth"
    model_path = "ESRGAN/models/2x_MangaScaleV3.pth"
    if not os.path.exists(model_path):
        print("⬇️ Downloading 2x_MangaScaleV3 model...")
        os.makedirs("ESRGAN/models", exist_ok=True)
        subprocess.run(["wget", "-O", model_path, model_url], check=True)

def prepare_dirs():
    os.makedirs(OUTPUT_FOLDER, exist_ok=True)
    os.makedirs("/content/temp_input", exist_ok=True)
    os.makedirs("/content/temp_output", exist_ok=True)

def get_all_images(folder):
    exts = [".jpg", ".jpeg", ".png", ".bmp", ".webp"]
    return [p for p in Path(folder).rglob("*") if p.suffix.lower() in exts]

def run_esrgan_batch(input_folder, output_folder, model_path):
    subprocess.run([
        "python", "ESRGAN/upscale.py", "-se",
        "-i", input_folder,
        "-o", output_folder,
        model_path
    ], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

def upscale_batch(batch, model_path, folder_root):
    temp_input = "/content/temp_input"
    temp_output = "/content/temp_output"

    for folder in [temp_input, temp_output]:
        for file in Path(folder).glob("*"):
            file.unlink()

    failed = []
    for img_path, rel_subfolder in batch:
        shutil.copy(img_path, temp_input)

    run_esrgan_batch(temp_input, temp_output, model_path)

    for img_path, rel_subfolder in batch:
        filename = img_path.name
        upscaled_path = os.path.join(temp_output, filename)
        out_dir = os.path.join(OUTPUT_FOLDER, rel_subfolder)
        os.makedirs(out_dir, exist_ok=True)
        final_path = os.path.join(out_dir, filename)

        if os.path.exists(upscaled_path):
            shutil.copy(upscaled_path, final_path)
        else:
            failed.append((img_path, rel_subfolder))

    return failed

def process_folder(folder_path, model_path):
    print(f"\n📁 Processing: {folder_path}")
    images = get_all_images(folder_path)
    total = len(images)
    batch_size = 5
    failed_all = []

    for i in range(0, total, batch_size):
        batch = images[i:i + batch_size]
        batch_data = [(img_path, str(img_path.parent.relative_to(folder_path))) for img_path in batch]

        start = time.time()
        failed = upscale_batch(batch_data, model_path, folder_path)
        end = time.time()

        print(f"🖼️ Batch {i+1}-{i+len(batch)} | ⏱ {end-start:.2f}s | ✅ {len(batch)-len(failed)} / ❌ {len(failed)}")

        failed_all.extend(failed)

    if failed_all:
        print(f"\n🔁 Retrying {len(failed_all)} failed image(s)...")
        retry_failed = upscale_batch(failed_all, model_path, folder_path)
        if retry_failed:
            print(f"❌ Still failed after retry ({len(retry_failed)}):")
            for path, _ in retry_failed:
                print(f" - {path}")
        else:
            print("✅ All failed images succeeded on retry.")
    else:
        print("✅ No failures.")

def main():
    print("🚀 Manga Upscaler (MangaScaleV3 Batch Mode)")
    if not torch.cuda.is_available():
        print("❌ Enable GPU: Runtime → Change runtime type → GPU")
        return

    mount_drive()
    clone_esrgan()
    download_model()
    prepare_dirs()

    if get_all_images(INPUT_FOLDER):
        process_folder(INPUT_FOLDER, "ESRGAN/models/2x_MangaScaleV3.pth")
        print(f"\n📦 Done! Upscaled images saved to:\n👉 {OUTPUT_FOLDER}")
    else:
        print("⚠️ No images found in the input folder.")

if __name__ == "__main__":
    main()


In [ ]:
# @title ⚡ Colab GPU Downscaler by Chapter (Auto-Resize Safe)

import os
from pathlib import Path
from PIL import Image
from tqdm import tqdm
import torch
import torchvision.transforms.functional as TF
from concurrent.futures import ThreadPoolExecutor

# --- CONFIG ---
input_folder = "/content/gdrive/MyDrive/render_tempt"  # @param {type:"string"}
output_folder = "/content/gd"  # @param {type:"string"}
batch_size = 6  # @param {type:"integer"}
max_workers = 2  # @param {type:"integer"}

exts = ['.jpg', '.jpeg', '.png', '.bmp', '.webp']
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"🖥️ Using device: {device}")

# --- Group Images by Chapter Folder ---
from collections import defaultdict
chapter_groups = defaultdict(list)

for root, _, files in os.walk(input_folder):
    for file in files:
        if Path(file).suffix.lower() in exts:
            full_path = os.path.join(root, file)
            chapter = os.path.relpath(root, input_folder)
            chapter_groups[chapter].append(full_path)

# --- Chunk Helper ---
def chunk(lst, n):
    for i in range(0, len(lst), n):
        yield lst[i:i + n]

# --- Process a Single Batch (with resizing) ---
def process_batch(batch_paths):
    tensors, rel_paths = [], []
    ref_size = None

    for path in batch_paths:
        try:
            with Image.open(path).convert("RGB") as img:
                tensor = TF.to_tensor(img)
                h, w = tensor.shape[1], tensor.shape[2]

                if ref_size is None:
                    ref_size = (h, w)
                elif (h, w) != ref_size:
                    img = img.resize(ref_size, Image.BICUBIC)
                    tensor = TF.to_tensor(img)

                tensors.append(tensor)
                rel_paths.append(os.path.relpath(path, input_folder))

        except Exception as e:
            print(f"❌ Load failed: {path} — {e}")

    if not tensors:
        return

    try:
        batch_tensor = torch.stack(tensors).to(device)
        new_size = (ref_size[0] // 2, ref_size[1] // 2)

        resized = torch.nn.functional.interpolate(
            batch_tensor, size=new_size, mode='bicubic', align_corners=False)

        for i, rel_path in enumerate(rel_paths):
            out_path = os.path.join(output_folder, rel_path)
            os.makedirs(os.path.dirname(out_path), exist_ok=True)
            img_out = TF.to_pil_image(resized[i].cpu())
            img_out.save(out_path)

    except RuntimeError as e:
        print(f"🚨 GPU error, fallback to single: {e}")
        for path in batch_paths:
            try:
                with Image.open(path).convert("RGB") as img:
                    tensor = TF.to_tensor(img).unsqueeze(0).to(device)
                    h, w = tensor.shape[2], tensor.shape[3]
                    new_size = (h // 2, w // 2)

                    resized = torch.nn.functional.interpolate(
                        tensor, size=new_size, mode='bicubic', align_corners=False)

                    rel_path = os.path.relpath(path, input_folder)
                    out_path = os.path.join(output_folder, rel_path)
                    os.makedirs(os.path.dirname(out_path), exist_ok=True)
                    img_out = TF.to_pil_image(resized[0].cpu())
                    img_out.save(out_path)

            except Exception as e2:
                print(f"❌ Fallback failed for {path}: {e2}")

# --- Process All Chapters ---
all_batches = []

for chapter, paths in chapter_groups.items():
    batches = list(chunk(paths, batch_size))
    all_batches.extend(batches)

print(f"📚 Found {sum(len(v) for v in chapter_groups.values())} images in {len(all_batches)} total batches across {len(chapter_groups)} chapters.")

with ThreadPoolExecutor(max_workers=max_workers) as executor:
    list(tqdm(executor.map(process_batch, all_batches), total=len(all_batches), desc="⚡ Downscaling", ncols=80))

print("✅ All chapter images processed and saved with folder structure preserved.")


# AI Manga Upscale Colab GITHUB



In [ ]:
# @title Manga Upscaler { display-mode: "form" }

import os
from google.colab import drive
import subprocess
from pathlib import Path
import torch

def check_connect_gdrive():
  if not os.path.exists("/content/gdrive/MyDrive"):
    print("Google Drive connection in progress...")
    drive.mount("/content/gdrive")

def check_clone_esrgan():
  if not os.path.exists("ESRGAN"):
    print("Downloading ESRGAN along with the AI models...")
    subprocess.run(["git", "clone", "https://github.com/Spladenly/ESRGAN"])
def init_dirs():
  Path("/content/gdrive/MyDrive/ESRGAN").mkdir \
   (parents=True, exist_ok=True)
  Path("/content/gdrive/MyDrive/ESRGAN/bw").mkdir \
   (parents=True, exist_ok=True)
  Path("/content/gdrive/MyDrive/ESRGAN/color").mkdir \
   (parents=True, exist_ok=True)
  Path("/content/gdrive/MyDrive/ESRGAN/output").mkdir \
   (parents=True, exist_ok=True)

def dir_contains_files(path):
  for root, dirs, files in os.walk(path):
    if files:
      return True
  return False

def ai_process_bw():
  !python ESRGAN/upscale.py -se -i /content/gdrive/MyDrive/ESRGAN/bw \
  -o /content/gdrive/MyDrive/ESRGAN/output \
  ESRGAN/models/4x_eula_digimanga_bw_v2_nc1_307k.pth

def ai_process_color():
  !python ESRGAN/upscale.py -se -i /content/gdrive/MyDrive/ESRGAN/color \
  -o /content/gdrive/MyDrive/ESRGAN/output \
  ESRGAN/models/4x-AnimeSharp.pth

def main():
  print("[AI Manga Upscale Colab] Manga Upscaler")

  if not torch.cuda.is_available():
    print("This session doesn't have a GPU.\n To connect a GPU, click:\n'Edit' -> 'Notebook settings' -> 'Hardware accelerator' = GPU; 'GPU type' = T4.\nAfter that, run this script again.")
    return

  check_connect_gdrive()
  check_clone_esrgan()
  init_dirs()

  status = 0

  if dir_contains_files("/content/gdrive/MyDrive/ESRGAN/bw"):
    status += 1
    print("Upscaling bw...")
    ai_process_bw()

  if dir_contains_files("/content/gdrive/MyDrive/ESRGAN/color"):
    status += 1
    print("Upscaling color...")
    ai_process_color()

  if status == 0:
    print("No pages were found in the following directories: '/ESRGAN/bw' and '/ESRGAN/color' on your Google Drive.\nPlease upload manga pages there, and run this script again.")
  else:
    print("The processing has been finished. The result can be downloaded from '/ESRGAN/output' on your Google Drive.\nTo process additional pages, run this script again. If you don't plan to process additional pages in the near future, please close the current session.\nThis can be done by clicking on: the inverted triangle (next to the 'RAM' and 'Disk' labels in the upper right corner) -> 'Disable and remove runtime'.\nThis will unlock the resources reserved for this session for other users.")

if __name__ == "__main__":
  main()

You may also upload CBZs and ZIPs instead of images. Use the following script to extract them before running Upscale Manga. After the extraction, the original files will be moved to recycle bin.

In [ ]:
# @title Extractor { display-mode: "form" }

import os
import shutil
import zipfile
from google.colab import drive

def check_connect_gdrive():
  if not os.path.exists("/content/gdrive/MyDrive"):
    print("Google Drive connection in progress...")
    drive.mount("/content/gdrive")

def unpack(path):
  zip_files = [f for f in os.listdir(path) if f.endswith(".zip") or \
               f.endswith(".cbz")]

  print(f"Found {len(zip_files)} file(s).")

  for zip_file in zip_files:
      print(f"Extracting {zip_file}...")

      zip_path = os.path.join(path, zip_file)

      if not os.path.exists(os.path.splitext(zip_path)[0]):
        extract_folder = os.path.join(path, \
                                        os.path.splitext(zip_file)[0])
        os.makedirs(extract_folder, exist_ok=True)

        with zipfile.ZipFile(zip_path, "r") as zf:
            zf.extractall(extract_folder)

        os.remove(zip_path)
      else:
         print("The folder already exists -- skipping.")

def main():
  print("[AI Manga Upscale Colab] Extractor")

  check_connect_gdrive()

  status = 0
  bw_path = "/content/gdrive/MyDrive/ESRGAN/bw"
  color_path = "/content/gdrive/MyDrive/ESRGAN/color"

  if os.path.exists(bw_path):
    status += 1
    print(f"Scanning: {bw_path}")
    unpack(bw_path)

  if os.path.exists(color_path):
    status += 1
    print(f"Scanning: {color_path}")
    unpack(color_path)

  if status == 0:
    print("Both 'bw' and 'color' directories don't exist.")

  print("Done.")

if __name__ == "__main__":
  main()